# RAG Pipeline Comparison Notebook

This notebook mirrors the retrieval path from `api/app.py` and helps you compare:
- Normal RAG (semantic retrieval only)
- Advanced RAG (metadata filtering + cross-encoder reranking)

It uses the same building blocks as the app:
- Query classifier (`api/query_classifier.py`)
- Astra vector stores (`finance`, `marketing`)
- Cross-encoder reranker (`api/cross_encoder.py`)
- ChatGroq generation model (`llama-3.3-70b-versatile`)

In [1]:
import os
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import InferenceClient
from langchain_astradb import AstraDBVectorStore
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from pydantic import SecretStr

from api.query_classifier import QueryClassifier
from api.cross_encoder import CrossEncoderReranker

load_dotenv()

print('Environment loaded')

/Users/nitpatel/Coding/multi_tenant_system/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment loaded


In [2]:
RAG_FETCH_K = int(os.getenv('RAG_FETCH_K', '15'))
RAG_USE_TOP_K = int(os.getenv('RAG_USE_TOP_K', '8'))
RAG_RERANK_MAX = int(os.getenv('RAG_RERANK_MAX', '15'))

EMBED_MODEL = 'sentence-transformers/all-mpnet-base-v2'
GEN_MODEL = 'llama-3.3-70b-versatile'

required = [
    'HF_TOKEN',
    'GROQ_API_KEY',
    'ASTRA_DB_API_ENDPOINT',
    'ASTRA_DB_APPLICATION_TOKEN',
    'ASTRA_DB_NAMESPACE',
]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise ValueError(f'Missing required env vars: {missing}')

print('Config OK')
print({'fetch_k': RAG_FETCH_K, 'top_k': RAG_USE_TOP_K, 'rerank_max': RAG_RERANK_MAX})

Config OK
{'fetch_k': 15, 'top_k': 8, 'rerank_max': 15}


In [3]:
class RouterHuggingFaceEmbeddings(Embeddings):
    def __init__(self, api_key: str, model_name: str) -> None:
        if not api_key:
            raise ValueError('HF_TOKEN is required for endpoint embeddings.')
        self._client = InferenceClient(model=model_name, token=api_key)

    def embed_documents(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        vectors = []
        for text in texts:
            result = self._client.feature_extraction(text)
            vectors.append(result)
        return vectors

    def embed_query(self, text):
        return self.embed_documents([text])[0]


def get_astra_config() -> Dict[str, str]:
    return {
        'api_endpoint': os.getenv('ASTRA_DB_API_ENDPOINT'),
        'token': os.getenv('ASTRA_DB_APPLICATION_TOKEN'),
        'namespace': os.getenv('ASTRA_DB_NAMESPACE'),
    }


def init_vectorstores(embedding_model: Embeddings) -> Dict[str, AstraDBVectorStore]:
    cfg = get_astra_config()
    stores = {}
    for collection in ['finance', 'marketing']:
        stores[collection] = AstraDBVectorStore(
            embedding=embedding_model,
            api_endpoint=cfg['api_endpoint'],
            token=cfg['token'],
            namespace=cfg['namespace'],
            collection_name=collection,
        )
    return stores


embeddings = RouterHuggingFaceEmbeddings(
    api_key=os.getenv('HF_TOKEN'),
    model_name=EMBED_MODEL,
)

vectorstores = init_vectorstores(embeddings)
query_classifier = QueryClassifier()
reranker = CrossEncoderReranker()

print('Initialized collections:', list(vectorstores.keys()))
print('Reranker model:', reranker.model_name)

Initialized collections: ['finance', 'marketing']
Reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2


In [4]:
def safe_get(doc, key, default=None):
    if hasattr(doc, 'metadata') or hasattr(doc, 'page_content'):
        if key == 'metadata':
            return getattr(doc, 'metadata', default)
        if key == 'page_content':
            return getattr(doc, 'page_content', default)
        return getattr(doc, key, default)
    if isinstance(doc, dict):
        return doc.get(key, default)
    return default


def compact_source_view(docs: List, label: str) -> pd.DataFrame:
    rows = []
    for i, doc in enumerate(docs, start=1):
        meta = safe_get(doc, 'metadata', {}) or {}
        content = (safe_get(doc, 'page_content', '') or '').replace('\n', ' ')
        rows.append({
            'rank': i,
            'pipeline': label,
            'collection': meta.get('collection', 'unknown'),
            'book_name': meta.get('book_name', ''),
            'author': meta.get('author', ''),
            'preview': content[:180],
        })
    return pd.DataFrame(rows)


def get_doc_key(doc) -> str:
    meta = safe_get(doc, 'metadata', {}) or {}
    content = safe_get(doc, 'page_content', '') or ''
    return '|'.join([
        str(meta.get('book_name', '')),
        str(meta.get('author', '')),
        str(meta.get('filename', '')),
        content[:120],
    ])


def summarize_comparison(normal_docs: List, advanced_docs: List) -> pd.DataFrame:
    normal_keys = {get_doc_key(d) for d in normal_docs}
    advanced_keys = {get_doc_key(d) for d in advanced_docs}

    overlap = len(normal_keys.intersection(advanced_keys))
    only_normal = len(normal_keys - advanced_keys)
    only_advanced = len(advanced_keys - normal_keys)

    normal_books = len({(safe_get(d, 'metadata', {}) or {}).get('book_name', '') for d in normal_docs})
    advanced_books = len({(safe_get(d, 'metadata', {}) or {}).get('book_name', '') for d in advanced_docs})

    return pd.DataFrame([
        {'metric': 'normal_count', 'value': len(normal_docs)},
        {'metric': 'advanced_count', 'value': len(advanced_docs)},
        {'metric': 'overlap_count', 'value': overlap},
        {'metric': 'only_normal', 'value': only_normal},
        {'metric': 'only_advanced', 'value': only_advanced},
        {'metric': 'normal_unique_books', 'value': normal_books},
        {'metric': 'advanced_unique_books', 'value': advanced_books},
    ])

In [5]:
@dataclass
class RetrievalTrace:
    query: str
    category: str
    confidence: float
    collection: str
    book_filter: Optional[str]
    semantic_count: int
    output_count: int
    latency_sec: float


def classify_collection(query: str, forced_collection: Optional[str] = None) -> Tuple[str, float, str]:
    if forced_collection:
        return 'forced', 1.0, forced_collection

    category, confidence = query_classifier.classify_query(query)
    collection_name = query_classifier.get_collection_name(category)

    if collection_name not in vectorstores:
        collection_name = 'finance'

    return category, confidence, collection_name


def retrieve_normal_rag(query: str, forced_collection: Optional[str] = None, k: int = RAG_USE_TOP_K):
    t0 = time.time()
    category, confidence, collection_name = classify_collection(query, forced_collection)

    retriever = vectorstores[collection_name].as_retriever(
        search_type='similarity',
        search_kwargs={'k': RAG_FETCH_K},
    )
    docs_sem = retriever.invoke(query)
    docs = docs_sem[:k]

    trace = RetrievalTrace(
        query=query,
        category=category,
        confidence=confidence,
        collection=collection_name,
        book_filter=None,
        semantic_count=len(docs_sem),
        output_count=len(docs),
        latency_sec=round(time.time() - t0, 3),
    )
    return docs, trace


def retrieve_advanced_rag(
    query: str,
    forced_collection: Optional[str] = None,
    book_filter: Optional[str] = None,
    k: int = RAG_USE_TOP_K,
):
    t0 = time.time()
    category, confidence, collection_name = classify_collection(query, forced_collection)

    search_kwargs = {'k': RAG_FETCH_K}
    if book_filter:
        search_kwargs['filter'] = {'book_name': book_filter}

    retriever = vectorstores[collection_name].as_retriever(
        search_type='similarity',
        search_kwargs=search_kwargs,
    )

    docs_sem = retriever.invoke(query)
    docs_to_rerank = docs_sem[:RAG_RERANK_MAX]

    if not docs_to_rerank:
        ranked = []
    else:
        ranked = reranker.rerank(query, docs_to_rerank, top_k=k)

    trace = RetrievalTrace(
        query=query,
        category=category,
        confidence=confidence,
        collection=collection_name,
        book_filter=book_filter,
        semantic_count=len(docs_sem),
        output_count=len(ranked),
        latency_sec=round(time.time() - t0, 3),
    )
    return ranked, trace

## Single Query Walkthrough

You can change the query and optional forced book filter to see the retrieval difference.

In [6]:
query = 'What does Warren Buffett say about margin of safety and intrinsic value?'

is_book_specific, detected_book = query_classifier.detect_book_specific_query(query)
book_filter = detected_book if is_book_specific else None

normal_docs, normal_trace = retrieve_normal_rag(query)
advanced_docs, advanced_trace = retrieve_advanced_rag(query, book_filter=book_filter)

print('Normal trace:', normal_trace)
print('Advanced trace:', advanced_trace)

summary_df = summarize_comparison(normal_docs, advanced_docs)
summary_df

Normal trace: RetrievalTrace(query='What does Warren Buffett say about margin of safety and intrinsic value?', category='finance', confidence=0.95, collection='finance', book_filter=None, semantic_count=15, output_count=8, latency_sec=3.488)
Advanced trace: RetrievalTrace(query='What does Warren Buffett say about margin of safety and intrinsic value?', category='finance', confidence=0.95, collection='finance', book_filter=None, semantic_count=15, output_count=8, latency_sec=13.783)


,metric,value
0,normal_count,8
1,advanced_count,8
2,overlap_count,7
3,only_normal,1
4,only_advanced,1
5,normal_unique_books,4
6,advanced_unique_books,4


In [7]:
normal_df = compact_source_view(normal_docs, 'normal')
advanced_df = compact_source_view(advanced_docs, 'advanced')

print('Top sources from normal RAG')
display(normal_df)

print('Top sources from advanced RAG')
display(advanced_df)

Top sources from normal RAG


,rank,pipeline,collection,book_name,author,preview
0,1,normal,unknown,schroeder_the-snowball-,unknown,Dodd stressed that there is no single definiti...
1,2,normal,unknown,the intelligent investor - benjamin graham,unknown,"uncharted world. With Graham as your guide, yo..."
2,3,normal,unknown,schroeder_the-snowball-,unknown,enabled him to invest in the market against hi...
3,4,normal,unknown,the intelligent investor - benjamin graham,unknown,in concentrating their purchases in the upper ...
4,5,normal,unknown,schroeder_the-snowball-,unknown,"margin of safety, the circle of competence, Mr..."
5,6,normal,unknown,poor charlie’s almanack_ the wit and wisdom of...,unknown,But that just related to Berkshire's intrinsic...
6,7,normal,unknown,the intelligent investor - benjamin graham,unknown,of safety. Thus the very class of “fair-weathe...
7,8,normal,unknown,a random walk down wall street,burton malkiel,theory relies on some tricky forecasts of the ...


Top sources from advanced RAG


,rank,pipeline,collection,book_name,author,preview
0,1,advanced,unknown,schroeder_the-snowball-,unknown,Dodd stressed that there is no single definiti...
1,2,advanced,unknown,the intelligent investor - benjamin graham,unknown,"uncharted world. With Graham as your guide, yo..."
2,3,advanced,unknown,poor charlie’s almanack_ the wit and wisdom of...,unknown,But that just related to Berkshire's intrinsic...
3,4,advanced,unknown,the intelligent investor - benjamin graham,unknown,in concentrating their purchases in the upper ...
4,5,advanced,unknown,schroeder_the-snowball-,unknown,enabled him to invest in the market against hi...
5,6,advanced,unknown,schroeder_the-snowball-,unknown,"margin of safety, the circle of competence, Mr..."
6,7,advanced,unknown,a random walk down wall street,burton malkiel,theory relies on some tricky forecasts of the ...
7,8,advanced,unknown,schroeder_the-snowball-,unknown,with the conscious intention of working on Wal...


## Multi-Query Evaluation

Run multiple test questions and compare latency plus how much the ranked set changes.

In [8]:
test_queries = [
    'How do I think about intrinsic value as a long-term investor?',
    'What is the role of diversification according to Buffett-style investing?',
    'Give me practical advice to avoid emotional decisions when markets fall.',
    'What are the key lessons from Rich Dad Poor Dad on assets and liabilities?',
    'How can I improve persuasion in sales conversations?',
]

rows = []
for q in test_queries:
    is_book_specific, detected_book = query_classifier.detect_book_specific_query(q)
    book_filter = detected_book if is_book_specific else None

    ndocs, ntrace = retrieve_normal_rag(q)
    adocs, atrace = retrieve_advanced_rag(q, book_filter=book_filter)

    nkeys = {get_doc_key(d) for d in ndocs}
    akeys = {get_doc_key(d) for d in adocs}

    rows.append({
        'query': q,
        'collection': atrace.collection,
        'book_filter': book_filter or '',
        'normal_latency_sec': ntrace.latency_sec,
        'advanced_latency_sec': atrace.latency_sec,
        'normal_docs': len(ndocs),
        'advanced_docs': len(adocs),
        'overlap_docs': len(nkeys.intersection(akeys)),
        'only_in_advanced': len(akeys - nkeys),
    })

eval_df = pd.DataFrame(rows)
eval_df

,query,collection,book_filter,normal_latency_sec,advanced_latency_sec,normal_docs,advanced_docs,overlap_docs,only_in_advanced
0,How do I think about intrinsic value as a long...,finance,,14.752,8.602,8,8,7,1
1,What is the role of diversification according ...,finance,,8.768,8.511,8,8,7,1
2,Give me practical advice to avoid emotional de...,finance,,8.787,5.213,8,8,6,2
3,What are the key lessons from Rich Dad Poor Da...,finance,rich dad poor dad,7.722,28.495,8,8,6,2
4,How can I improve persuasion in sales conversa...,finance,,22.144,7.834,8,8,7,1


In [9]:
agg = pd.DataFrame([
    {'metric': 'avg_normal_latency_sec', 'value': round(eval_df['normal_latency_sec'].mean(), 3)},
    {'metric': 'avg_advanced_latency_sec', 'value': round(eval_df['advanced_latency_sec'].mean(), 3)},
    {'metric': 'avg_overlap_docs', 'value': round(eval_df['overlap_docs'].mean(), 2)},
    {'metric': 'avg_only_in_advanced', 'value': round(eval_df['only_in_advanced'].mean(), 2)},
])
agg

,metric,value
0,avg_normal_latency_sec,12.435
1,avg_advanced_latency_sec,11.731
2,avg_overlap_docs,6.600
3,avg_only_in_advanced,1.400


## Optional: Compare Final Generated Answers

This uses the same generation model configured in your app (`llama-3.3-70b-versatile`) and shows how retrieval differences affect final answer quality.

In [10]:
llm = ChatGroq(
    api_key=SecretStr(os.getenv('GROQ_API_KEY')),
    model=GEN_MODEL,
    temperature=0.3,
    max_tokens=700,
)

template = '''
You are a senior financial mentor speaking to your disciple.
Answer directly, clearly, and practically.

Context:
{context}

User:
{question}
'''
prompt = PromptTemplate(template=template, input_variables=['context', 'question'])
chain = prompt | llm | StrOutputParser()

def docs_to_context(docs: List) -> str:
    chunks = []
    for d in docs:
        meta = safe_get(d, 'metadata', {}) or {}
        text = safe_get(d, 'page_content', '') or ''
        chunks.append(f"Book: {meta.get('book_name', '')} | Author: {meta.get('author', '')}\n{text}")
    return '\n\n'.join(chunks)

q = query
normal_answer = chain.invoke({'context': docs_to_context(normal_docs), 'question': q})
advanced_answer = chain.invoke({'context': docs_to_context(advanced_docs), 'question': q})

print('NORMAL RAG ANSWER')
print(normal_answer)
print('\n' + '=' * 100 + '\n')
print('ADVANCED RAG ANSWER')
print(advanced_answer)

NORMAL RAG ANSWER
Warren Buffett, a disciple of Benjamin Graham, emphasizes the importance of "margin of safety" in investing. He believes that the margin of safety is the difference between the intrinsic value of a stock and its market price. In other words, it's the buffer between what a stock is worth and what you pay for it.

Buffett has said that he looks for a significant margin of safety when investing, which means he wants to buy stocks at a price that is substantially lower than their intrinsic value. This approach helps to protect him from losses if the stock's price falls or if his estimates of its intrinsic value are incorrect.

In terms of intrinsic value, Buffett defines it as the present value of all future cash flows that a business is expected to generate. He considers various factors such as earnings, dividends, assets, capital structure, and other factors to estimate a company's intrinsic value.

Buffett's approach to margin of safety and intrinsic value is rooted in

## Notes

- If retrieval is empty, verify Astra credentials and collection names.
- Advanced RAG is usually slower than normal RAG because reranking is extra work.
- The reranker in this project is `cross-encoder/ms-marco-MiniLM-L-6-v2` (via HF Inference API).
- The generation model used in this notebook and your app is `llama-3.3-70b-versatile`.